# Semantic Router Evaluation Pipeline

This notebook evaluates the `SemanticRouterModule` performance. It classifies legal queries into **Reasoning-LLM**, **General-LLM**, or **Casual-LLM** routes.

**Current Task:** Benchmarking between Qwen-2.5 and Gemini-2.5 Flash Lite

In [1]:
# 1. Install Dependencies
%pip install pandas tqdm requests python-dotenv rich scikit-learn -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ==================================================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==================================================================
import sys, os, time, json, re, types
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from sklearn.metrics import classification_report

# --- Path Management ---
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path: sys.path.append(project_root)
load_dotenv(os.path.join(project_root, '.env'))

# --- Framework Imports ---
from src.adaptive_routing.modules.triage import TriageModule
from src.adaptive_routing.modules.router import SemanticRouterModule
from src.adaptive_routing.config import FrameworkConfig

# --- Settings ---
OPENROUTER_API_KEY = ""  # Leave empty to use .env file
QUERY_LIMIT = None # Set to None for full evaluation
SELECTED_MODEL = "Gemini" # @param ["Qwen-2.5", "Gemini-2.5"]

MODEL_OPTIONS = {
    "Qwen": "qwen/qwen-2.5-7b-instruct",
    "Gemini": "google/gemini-2.5-flash-lite"
}

CONFIG = {
    "api_key": OPENROUTER_API_KEY or os.getenv("OPENROUTER_API_KEY"),
    "router_model": MODEL_OPTIONS[SELECTED_MODEL],
    "router_temp": 0.1,                   
    "router_max_tokens": 250,         
    "router_use_system": True
}

SYSTEM_PROMPT = ""

# --- Initialization ---
FrameworkConfig._update_settings_(**CONFIG)
triage = TriageModule(api_key=CONFIG['api_key'])
router = SemanticRouterModule(api_key=CONFIG['api_key'])


print(f"Engine ready with model: {FrameworkConfig._ROUTER_MODEL}")

c:\Users\Prince\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Engine ready with model: google/gemini-2.5-flash-lite


In [2]:
# ==================================================================
# 2. LOAD DATASET & MANAGE CHECKPOINTS
# ==================================================================
dataset_path = 'dataset/Routing-Evaluation-Dataset.csv'
input_column = 'Query'
label_column = 'Groq Expected'
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

# Load or Resume from Checkpoint
if os.path.exists(checkpoint_path):
    print(f"Resuming from checkpoint: {checkpoint_path}")
    df = pd.read_csv(checkpoint_path)
    if pred_col not in df.columns:
        df[pred_col] = None
else:
    print(f"Loading fresh dataset: {dataset_path}")
    df = pd.read_csv(dataset_path)
    df[pred_col] = None

# Apply Query Limit if set in configurations
if QUERY_LIMIT:
    df = df.head(QUERY_LIMIT)

print(f"Dataset Ready: {len(df)} rows loaded.")

Loading fresh dataset: dataset/Routing-Evaluation-Dataset.csv
Dataset Ready: 581 rows loaded.


In [3]:
# ==================================================================
# 3. QUICK TEST (Single Inference)
# ==================================================================
sample_query = "How do I file a case for illegal dismissal?"

print(f"Testing: '{sample_query}'")
try:
    start_time = time.time()
    triage_res = triage._process_request_(sample_query)
    norm_text = triage_res.get("normalized_text", sample_query)
    
    route_res = router._process_routing_(norm_text)
    duration = time.time() - start_time
    
    print("\n--- RESULTS ---")
    print(f"Detected Route: {route_res.get('route')}")
    print(f"Confidence: {route_res.get('confidence')}")
    print(f"Latency: {duration:.2f} seconds")
except Exception as e:
    print(f"Error: {e}")

Testing: 'How do I file a case for illegal dismissal?'


Language tag not found in normalizer output. First 100 chars: To file a case for **illegal dismissal**, you must follow the legal procedures outlined in the **Lab



--- RESULTS ---
Detected Route: General-LLM
Confidence: 0.95
Latency: 15.90 seconds


In [4]:
# ==================================================================
# 4. EXECUTE EVALUATION (With Retry Logic & Increment Logic)
# ==================================================================
pred_col = f"Predicted_{SELECTED_MODEL}"
checkpoint_path = f'dataset/Routing-Evaluation-Checkpoint-{SELECTED_MODEL}.csv'

print(f"Starting evaluation for {SELECTED_MODEL}...")
for index, row in tqdm(df.iterrows(), total=len(df)):
    current_val = row.get(pred_col)
    
    # --- INCREMENTAL LOGIC ---
    # Skip only if row has a valid non-error prediction
    if pd.notna(current_val) and str(current_val).strip() != "" and not str(current_val).startswith("ERROR"):
        continue
    
    # --- RETRY LOGIC (3 attempts) ---
    for attempt in range(1, 4):
        try:
            query = str(row[input_column])
            t_res = triage._process_request_(query)
            norm_text = t_res.get("normalized_text", query)
            r_res = router._process_routing_(norm_text)
            
            if r_res.get("route"):
                df.at[index, pred_col] = r_res.get('route')
                break # Success
            else:
                raise ValueError("Empty route returned")
                
        except Exception as e:
            if attempt == 3:
                df.at[index, pred_col] = f"ERROR: {str(e)[:50]}"
            else:
                time.sleep(2 ** attempt) # Exponential backoff
    
    if (index + 1) % 10 == 0: df.to_csv(checkpoint_path, index=False)

print("Evaluation cycle complete.")

Starting evaluation for Gemini...


  0%|          | 0/581 [00:00<?, ?it/s]Language tag not found in normalizer output. First 100 chars: **Pagsusuri at Pormal na Pagsasalin (Legal Linguistic Normalization):**

**Original Text (Tagalog):*
  1%|          | 7/581 [01:26<2:08:47, 13.46s/it]Language tag not found in normalizer output. First 100 chars: **Pangunahing Tugon (Pinoy English):**

"May rumors about pandemic in Hong Kong. How can I help my f
  3%|▎         | 15/581 [03:25<2:22:39, 15.12s/it]Language tag not found in normalizer output. First 100 chars: **Pangunahing Konklusyon:**

Ang pagkawala ng iyong Hong Kong Identity Document (HKID) sa Pilipinas 
  4%|▍         | 26/581 [05:41<1:35:56, 10.37s/it]Language tag not found in normalizer output. First 100 chars: **Sagot:**

Hindi ito maaaring gawin. Ang pagkakasama ng bayad sa holiday sa sahod ay **hindi tugma 
  5%|▍         | 27/581 [05:51<1:32:29, 10.02s/it]Language tag not found in normalizer output. First 100 chars: **Pagsusuri sa Tanong:**

Ang tanong ay tungkol 

KeyboardInterrupt: 

In [3]:
# 6. Summary & Results (Dual-Reference Evaluation)
import pandas as pd
import os
from sklearn.metrics import classification_report

# Load combined results for full comparison
combined_results_path = 'dataset/Routing-Evaluation-Results-Combine.csv'
if os.path.exists(combined_results_path):
    eval_df = pd.read_csv(combined_results_path)
else:
    eval_df = df.copy()

# References to compare against
reference_cols = ['Expected', 'Groq Expected']
# Models to evaluate
prediction_cols = ['Predicted_Gemini', 'Predicted_Qwen']

for ref in reference_cols:
    if ref not in eval_df.columns:
        print(f"\nReference column '{ref}' not found. Skipping...")
        continue
        
    print(f"\n{'#'*60}")
    print(f" EVALUATION AGAINST REFERENCE: {ref}")
    print(f"{'#'*60}")
    
    for pred in prediction_cols:
        if pred not in eval_df.columns:
            print(f"\nPrediction column '{pred}' not found in dataset. Skipping...")
            continue
            
        print(f"\n{'='*40}")
        print(f" Model: {pred}")
        print(f"{'='*40}")
        
        # Filter for valid rows: both columns not null and pred is not an ERROR string
        mask = (eval_df[ref].notna()) & (eval_df[pred].notna()) & \
               (~eval_df[pred].astype(str).str.startswith("ERROR", na=False))
        eval_subset = eval_df[mask]
        
        if not eval_subset.empty:
            accuracy = (eval_subset[pred] == eval_subset[ref]).mean() * 100
            print(f"Accuracy (vs {ref}): {accuracy:.2f}%")
            
            print(f"\nClassification Report ({pred} vs {ref}):")
            try:
                report = classification_report(
                    eval_subset[ref].astype(str), 
                    eval_subset[pred].astype(str), 
                    zero_division=0
                )
                print(report)
            except Exception as e:
                print(f"Could not generate report: {e}")
        else:
            print(f"No valid prediction data for {pred} to compare against {ref}.")

eval_df.head()


############################################################
 EVALUATION AGAINST REFERENCE: Expected
############################################################

 Model: Predicted_Gemini
Accuracy (vs Expected): 85.89%

Classification Report (Predicted_Gemini vs Expected):
               precision    recall  f1-score   support

   Casual-LLM       0.00      0.00      0.00         0
  General-LLM       0.85      0.88      0.86       287
Reasoning-LLM       0.88      0.84      0.86       294

     accuracy                           0.86       581
    macro avg       0.58      0.57      0.57       581
 weighted avg       0.87      0.86      0.86       581


 Model: Predicted_Qwen
Accuracy (vs Expected): 76.22%

Classification Report (Predicted_Qwen vs Expected):
               precision    recall  f1-score   support

   Casual-LLM       0.00      0.00      0.00         0
  General-LLM       0.77      0.75      0.76       285
Reasoning-LLM       0.76      0.78      0.77       287

     ac

,Query,Expected,Groq Expected,Predicted_Gemini,Predicted_Qwen
0,Tatlong OFW na nasa deathrow sa China... Naha...,Reasoning-LLM,Reasoning-LLM,Reasoning-LLM,Reasoning-LLM
1,Ang Passport and Employment Contract ko ay nas...,Reasoning-LLM,Reasoning-LLM,Reasoning-LLM,General-LLM
2,pag may pandemic. maiiwan ba ako here sa hongk...,General-LLM,Reasoning-LLM,Reasoning-LLM,Reasoning-LLM
3,Paano gumagana yung bagong Senate Bill No. 2390?,General-LLM,General-LLM,General-LLM,General-LLM
4,Nawalan ako ng trabaho dahil I made a Case sa ...,Reasoning-LLM,Reasoning-LLM,Reasoning-LLM,Reasoning-LLM
